# CayleyPy Megaminx — beam-search baseline using AZ v4 V-only

This notebook demonstrates a **minimal end-to-end submission pipeline** for the [CayleyPy Megaminx competition](https://www.kaggle.com/competitions/cayley-py-megaminx) using the **AZ v4 V-only** checkpoint from the [CayleyPy Megaminx Solvers](https://www.kaggle.com/models/artgor/cayleypy-megaminx-solvers) shared model collection. AZ v4 is a 6M-parameter dual-head Bellman + policy model trained with the m_dd_v0 anchor-mixin recipe; the V-only extraction is a drop-in `ResMLPDistance`.

**What this notebook does**

1. Loads AZ v4 V-only from the shared model mount.
2. Reads the competition `test.csv` and `sample_submission.csv`.
3. Defines a minimal beam-search solver in ~50 lines of PyTorch.
4. Runs the solver on the first N pids; for everything else, falls back to the sample submission.
5. Writes a valid `submission.csv` and reports stats.

**What this notebook does NOT do** (kept simple for clarity)

- No NISS, no Q-shortlister, no sym-ensemble, no multi-pass beam, no post-processing. These ship in the project repo (and in our paper) but would obscure the baseline pipeline.
- No hash-based duplicate filtering inside the beam. A larger beam width compensates for this on shallow puzzles.
- No fancy fallback. The competition's sample submission is the per-pid lower bound; for pids beam search doesn't solve, the row from `sample_submission.csv` is used verbatim.

**Expected runtime** on a Kaggle CPU notebook: ~10–15 minutes for N=15 at beam 1024, max_steps 100. Increase `N_PIDS` / `BEAM_WIDTH` if you switch to a GPU runtime (and pin a CUDA-compatible PyTorch wheel).


## Step 1. Imports and paths


In [ ]:
import os, sys, time, json, glob
import numpy as np
import pandas as pd
import torch

INPUT_ROOT = '/kaggle/input'
print('full /kaggle/input tree (depth <= 5):')
for path, dirs, files in os.walk(INPUT_ROOT):
    depth = path.replace(INPUT_ROOT, '').count(os.sep)
    indent = '  ' * depth
    print(f'{indent}{os.path.basename(path) or path}/')
    for f in sorted(files):
        print(f'{indent}  {f}')
    if depth >= 5:
        dirs[:] = []

def find_instance(slug: str) -> str:
    aliases = [slug, slug.replace('-', '_')]
    hits = []
    for path, dirs, files in os.walk(INPUT_ROOT):
        if os.path.basename(path) in aliases:
            numeric_children = sorted([d for d in dirs if d.isdigit()], key=int)
            if numeric_children:
                hits.append(os.path.join(path, numeric_children[-1]))
            elif files:
                hits.append(path)
    if not hits:
        for path, dirs, files in os.walk(INPUT_ROOT):
            if 'puzzle_info.json' in files and slug in path:
                hits.append(path)
    if not hits:
        raise RuntimeError(f'Could not locate variation {slug!r}')
    return hits[-1]

AZ_V4_PATH = find_instance('m-az-v4')
print()
print('AZ v4 instance path:', AZ_V4_PATH)
print('files:', sorted(os.listdir(AZ_V4_PATH)))

sys.path.insert(0, AZ_V4_PATH)
from model import ResMLPDistance
from puzzle import Megaminx

# This notebook uses CPU by default for portability across Kaggle runtimes.
# Switch to 'cuda' if you have a matching PyTorch wheel and a supported GPU.
device = 'cpu'
print('device:', device)


## Step 2. Load AZ v4 V-only

The extracted V head loads as a vanilla `ResMLPDistance(output_dim=1)`. The model_config is stored inline in the checkpoint so we don't have to remember the architecture hyperparameters.


In [ ]:
ckpt_path = os.path.join(AZ_V4_PATH, 'v_only.pt')
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
cfg = ckpt['model_config']
print('model_config:', cfg)

model = ResMLPDistance(**cfg, inference_chunk_size=8192)
sd = ckpt['state_dict']
if any(k.startswith('_orig_mod.') for k in sd):
    sd = {k.removeprefix('_orig_mod.'): v for k, v in sd.items()}
model.load_state_dict(sd)
model.to(device).eval()
print(f'loaded AZ v4 V-only: {model.num_parameters():,} parameters')

# Sanity check: V(solved) should be near 0 (AZ v4 was trained with the anchor mixin)
puzzle = Megaminx.load(os.path.join(AZ_V4_PATH, 'puzzle_info.json'))
solved_t = torch.tensor([list(puzzle.solved_state)], dtype=torch.long, device=device)
with torch.no_grad():
    v_solved = float(model(solved_t).item())
print(f'V(solved) = {v_solved:.4f}  (expected near 0 for AZ v4)')


## Step 3. Load competition data

`test.csv` has 1001 rows, each with `initial_state_id` (the pid) and `initial_state` (a comma-separated permutation of length 120). `sample_submission.csv` has the same id space and a `path` column (dot-separated move sequence) that defines our per-pid fallback floor.


In [ ]:
# Competition data may mount under one of several paths depending on Kaggle CLI version.
for cand in ['/kaggle/input/cayley-py-megaminx',
             '/kaggle/input/competitions/cayley-py-megaminx']:
    if os.path.isdir(cand):
        COMP_PATH = cand
        break
else:
    raise RuntimeError('competition data not mounted under any expected path')
print('competition files:', sorted(os.listdir(COMP_PATH)))

test_df = pd.read_csv(os.path.join(COMP_PATH, 'test.csv'))
sample_df = pd.read_csv(os.path.join(COMP_PATH, 'sample_submission.csv'))
print(f'test.csv: {len(test_df)} rows')
print(f'sample_submission.csv: {len(sample_df)} rows')

def parse_state(s: str) -> list[int]:
    return [int(x) for x in s.split(',')]

# Quick stats on sample-submission path lengths (these are upper bounds the model must beat)
sample_lens = sample_df['path'].apply(lambda p: len(p.split('.')))
print(f'sample submission: total moves = {int(sample_lens.sum()):,}, mean = {sample_lens.mean():.1f}')
print(f'sample submission: min = {sample_lens.min()}, max = {sample_lens.max()}')


## Step 4. Minimal beam-search solver

A neural beam search keeps the `B` lowest-V states per step. Each step:

1. **Expand**: for each of the B parents, generate all `|A|=24` children by applying every generator.
2. **Score**: run the V model on all `B × 24` children.
3. **Select**: keep the B with the smallest predicted distance.
4. **Check**: if any kept state equals V₀, walk back through stored parents to recover the move sequence.

This is the simplest possible version: no duplicate filtering, no NISS, no Q-shortlister. For real production runs you'd want all three.


In [ ]:
@torch.no_grad()
def beam_search(
    model,
    puzzle: Megaminx,
    initial_state: list[int],
    beam_width: int = 4096,
    max_steps: int = 120,
    device: str = 'cuda',
) -> tuple[bool, list[str]]:
    n_gen = len(puzzle.move_names)
    state_size = len(puzzle.solved_state)
    gens = torch.tensor([puzzle.generators[n] for n in puzzle.move_names],
                        dtype=torch.long, device=device)
    solved = torch.tensor(puzzle.solved_state, dtype=torch.long, device=device)
    move_names = list(puzzle.move_names)

    beam = torch.tensor([initial_state], dtype=torch.long, device=device)
    history = [(torch.full((1,), -1, dtype=torch.long, device=device),
                torch.full((1,), -1, dtype=torch.long, device=device))]

    for step in range(max_steps):
        eq = (beam == solved.unsqueeze(0)).all(dim=1)
        if eq.any():
            winner = int(eq.nonzero(as_tuple=False)[0].item())
            path = []
            idx = winner
            for parent_arr, move_arr in reversed(history):
                m = int(move_arr[idx].item())
                if m == -1:
                    break
                path.append(move_names[m])
                idx = int(parent_arr[idx].item())
            return True, list(reversed(path))

        B_cur = beam.shape[0]
        children = beam.unsqueeze(1).expand(B_cur, n_gen, state_size).gather(
            2, gens.unsqueeze(0).expand(B_cur, n_gen, state_size))
        children_flat = children.reshape(-1, state_size)
        parent_idx = torch.arange(B_cur, device=device).repeat_interleave(n_gen)
        move_idx = torch.arange(n_gen, device=device).repeat(B_cur)

        scores = model(children_flat)
        n_cand = children_flat.shape[0]
        k = min(beam_width, n_cand)
        keep = torch.topk(scores, k, largest=False).indices
        beam = children_flat[keep]
        history.append((parent_idx[keep], move_idx[keep]))

    return False, []


## Step 5. Sanity check on pid 1

Pid 1 has a 1-move scramble (the sample submission solves it in 1 move). Beam search at width 4096 should find this trivially.


In [ ]:
pid_1_state = parse_state(test_df.loc[test_df.initial_state_id == 1, 'initial_state'].iloc[0])
print(f'pid 1 differs from solved at {sum(1 for i, x in enumerate(pid_1_state) if x != i)} positions')

t0 = time.time()
found, path = beam_search(model, puzzle, pid_1_state, beam_width=1024, max_steps=40, device=device)
dt = time.time() - t0
print(f'beam search: found={found}, path_len={len(path)}, wall={dt:.1f}s')
print(f'path: {".".join(path)}')

# Verify by applying the path to the initial state
final = pid_1_state
for m in path:
    final = list(puzzle.apply_move(final, m))
assert final == list(puzzle.solved_state), 'verification failed!'
print('verified: applying the move sequence reaches V_0')


## Step 6. Run beam search on the first N pids

Pids are ordered roughly by scramble depth (pid 0 has a 72-move walk; pid 1000 has a 925-move walk), so the first N pids include the easy/medium difficulty range where a small-beam baseline is expected to perform well. Hard pids (later in the file) typically need much larger beams plus the inference tricks the baseline omits.

Set `N_PIDS` based on your time budget. At N=50, beam=4096, max_steps=120 on a P100 GPU, expect ~10–20 minutes total.


In [ ]:
N_PIDS = 15          # number of pids to solve with the model (rest use sample fallback)
BEAM_WIDTH = 1024    # CPU-friendly default; increase if you have GPU + matching PyTorch wheel
MAX_STEPS = 100      # cap on beam depth

results = []  # list of dicts with pid, found, path_len, beam_path, wall

for i in range(N_PIDS):
    pid = int(test_df.iloc[i]['initial_state_id'])
    init = parse_state(test_df.iloc[i]['initial_state'])
    t0 = time.time()
    found, path = beam_search(model, puzzle, init, beam_width=BEAM_WIDTH, max_steps=MAX_STEPS, device=device)
    dt = time.time() - t0
    sample_path = sample_df.loc[sample_df.initial_state_id == pid, 'path'].iloc[0]
    sample_len = len(sample_path.split('.'))
    if found:
        # Verify and compare
        final = init
        for m in path:
            final = list(puzzle.apply_move(final, m))
        verified = (final == list(puzzle.solved_state))
        marker = '*' if (len(path) < sample_len) else ' '
    else:
        verified = False
        marker = 'x'
    results.append({
        'pid': pid,
        'found': found,
        'verified': verified,
        'beam_len': len(path) if found else None,
        'sample_len': sample_len,
        'beam_path': '.'.join(path) if found else None,
        'wall_s': dt,
    })
    print(f'pid {pid:4d}  {marker}  found={found}  beam_len={len(path) if found else "-"}  sample_len={sample_len}  wall={dt:.1f}s')


## Step 7. Aggregate stats


In [ ]:
res_df = pd.DataFrame(results)
n_found = int(res_df['found'].sum())
n_verified = int(res_df['verified'].sum())
n_beat = int(((res_df['found']) & (res_df['beam_len'] < res_df['sample_len'])).sum())
total_beam = int(res_df.loc[res_df['found'], 'beam_len'].sum())
total_sample_for_solved = int(res_df.loc[res_df['found'], 'sample_len'].sum())
total_wall = res_df['wall_s'].sum()
print(f'solved by model: {n_found} / {N_PIDS} (verified: {n_verified})')
print(f'beam beat sample: {n_beat} / {n_found}')
print(f'total beam moves (on solved):   {total_beam:,}')
print(f'total sample moves (same set):  {total_sample_for_solved:,}')
print(f'savings on solved set:          {total_sample_for_solved - total_beam:,} moves')
print(f'total wall: {total_wall:.0f}s  ({total_wall / 60:.1f} min)')


## Step 8. Build the submission CSV

For pids 0..N-1: use the beam path if found and verified; else use the sample fallback. For pids N..1000: use the sample fallback verbatim. This guarantees a valid submission of the same length as `sample_submission.csv`.


In [ ]:
submission_rows = []
results_by_pid = {r['pid']: r for r in results}
for _, row in sample_df.iterrows():
    pid = int(row['initial_state_id'])
    fallback_path = row['path']
    r = results_by_pid.get(pid)
    if r is not None and r['found'] and r['verified'] and len(r['beam_path'].split('.')) <= len(fallback_path.split('.')):
        submission_rows.append({'initial_state_id': pid, 'path': r['beam_path']})
    else:
        submission_rows.append({'initial_state_id': pid, 'path': fallback_path})

submission_df = pd.DataFrame(submission_rows)
submission_df.to_csv('submission.csv', index=False)
print(f'wrote submission.csv with {len(submission_df)} rows')

total_moves = submission_df['path'].apply(lambda p: len(p.split('.'))).sum()
sample_total = sample_df['path'].apply(lambda p: len(p.split('.'))).sum()
print(f'submission total moves:  {int(total_moves):,}')
print(f'sample baseline:         {int(sample_total):,}')
print(f'reduction:               {int(sample_total - total_moves):,} moves on the {N_PIDS}-pid head')


## What to do next

This baseline produces a valid submission but does not approach the leaderboard top. To close that gap, layer the components the paper describes:

1. **Multi-pass beam + NISS** — `--beams 16384,65536 --max-steps 60,150 --niss` is the canonical production recipe. Single-pass fails for ~50% of pids past depth 60.
2. **Q-shortlister composition** — pair AZ v4 V with a Q-shortlister distilled from it. (The shared `m23_v2` was distilled from m05; pairing it with AZ v4 V regresses; see the m_az_v4 README for the caveat. We are not shipping an AZ v4-paired qshort yet.)
3. **Symmetry ensemble `K=4`** — apply the $A_5 \times C_6$ rotation table at inference: rotate the input state, run beam, translate the path back via the conjugation map, take the per-pid minimum. Requires the rotations table (small, derivable from the symmetry-derivation script in the project repo).
4. **Post-processing layers** — BFS-d6 sliding-window replacement (already applied to the competition's `sample_submission.csv` floor) plus tail re-solve plus simulated-annealing macro-insertion. Each saves a few hundred to a few thousand moves on a full-1001 submission.
5. **Per-pid min-merge across configurations** — different (beam width, hash seed, rotation) configurations find different shortcuts on different pids; their per-pid minimum is significantly stronger than any single configuration.

See the [companion notebook](https://www.kaggle.com/code/artgor/megaminx-models-quick-tour) for a tour of all 5 shared models, and the [model collection](https://www.kaggle.com/models/artgor/cayleypy-megaminx-solvers) for the other checkpoints (m05, m_curr_v3, m_dd_v0, m23_v2). Detailed training recipes and inference compositions are in the CayleyPy Megaminx paper.
